<a href="https://colab.research.google.com/github/thatcherty/ML-Algo-Selection/blob/main/ML_Algo_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Algo Selection

In [2]:
# Common
import pandas as pd
import numpy as np
from sklearn import datasets

# requires pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


# Logistic Regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# additional for Neural Network
# include MLP or CNN
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier


# Decision tree
from sklearn import tree

# Example fetching data

# fetch dataset
# we need to get the ID's of each dataset we want
# this will be a loop
istanbul_stock_exchange = fetch_ucirepo(id=247)

# each iteration will split data and assess against target algorithms
# each result will be entered into a panda dataframe

# data (as pandas dataframes)
X = istanbul_stock_exchange.data.features
y = istanbul_stock_exchange.data.targets

# metadata
print(istanbul_stock_exchange.metadata)

# variable information
print(istanbul_stock_exchange.variables)

{'uci_id': 247, 'name': 'ISTANBUL STOCK EXCHANGE', 'repository_url': 'https://archive.ics.uci.edu/dataset/247/istanbul+stock+exchange', 'data_url': 'https://archive.ics.uci.edu/static/public/247/data.csv', 'abstract': 'Data sets includes returns of Istanbul Stock Exchange with seven other international index; SP, DAX, FTSE, NIKKEI, BOVESPA, MSCE_EU, MSCI_EM from Jun 5, 2009 to Feb 22, 2011.', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate', 'Univariate', 'Time-Series'], 'num_instances': 536, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': None, 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2013, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C54P4J', 'creators': ['Oguz Akbilgic'], 'intro_paper': {'ID': 453, 'type': 'NATIVE', 'title': 'A novel Hybrid RBF Neural Networks model as a forecaster', 'authors': 'O. Akbilgic, H. Bozdogan, M.

In [17]:
import argparse
from pydoc import text
from pyexpat import features
import re
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill

DEFAULT_URL = (
    "https://archive.ics.uci.edu/datasets"
    "?Task=Classification&Python=true&skip=0&take=10&sort=desc&orderBy=NumHits&search="
)
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0 Safari/537.36"
)


def build_page_url(base_url: str, *, skip: int, take: int) -> str:
    parsed = urlparse(base_url)
    params = OrderedDict(parse_qsl(parsed.query, keep_blank_values=True))
    params["skip"] = str(skip)
    params["take"] = str(take)
    new_query = urlencode(list(params.items()), doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def extract_total_count(page_text: str) -> int | None:
    # Examples: "0 to 10 of 173" or "10 to 20 of 173"
    match = re.search(r"\b\d+\s+to\s+\d+\s+of\s+(\d+)\b", page_text)
    return int(match.group(1)) if match else None

def normalize_name(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def parse_dataset_links(html: str, page_url: str) -> list[int]:
    soup = BeautifulSoup(html, "html.parser")

    # Prefer title links that appear inside headings on the current UCI site.
    anchors = soup.select("h2 a[href]")

    # Fallback: any visible link pointing to a dataset page.
    if not anchors:
        anchors = soup.select("a[href*='/dataset/'], a[href*='/ml/datasets/']")

    dataset_id: list[int] = list()
    seen: set[tuple[str, str]] = set()

    for a in anchors:
        href = (a.get("href") or "").strip()
        name = normalize_name(a.get_text(" ", strip=True))

        if not href:
            continue
        if "/dataset/" not in href and "/ml/datasets/" not in href:
            continue
        if not name:
            continue

        full_url = urljoin(page_url, href)
        key = (name.casefold(), full_url)
        if key in seen:
            continue
        seen.add(key)
        # Extract the integer ID from the URL
        id_matches = re.findall(r'\d+', full_url)
        if id_matches:
            dataset_id.append(int(id_matches[0]))
    return dataset_id


def scrape_all_datasets(base_url: str, *, take: int = 25, timeout: int = 30) -> list[int]:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    all_ids: list[int] = list()
    seen_ids: set[int] = set()
    skip = 0
    total_count: int | None = None

    while True:
        page_url = build_page_url(base_url, skip=skip, take=take)
        response = session.get(page_url, timeout=timeout)
        response.raise_for_status()
        html = response.text
        text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

        if total_count is None:
            total_count = extract_total_count(text)

        page_ids = parse_dataset_links(html, page_url)
        page_new_rows = [id for id in page_ids if id not in seen_ids]

        if not page_new_rows:
            break

        for id in page_new_rows:
            seen_ids.add(id)
            all_ids.append(id)

        if total_count is not None and len(all_ids) >= total_count:
            break

        if len(page_ids) < take:
            break

        skip += take

    return all_ids



def get_data() -> list[int]:
    parser = argparse.ArgumentParser(
        description="Scrape UCI dataset names + links from a filtered listing page into Excel."
    )
    parser.add_argument("--url", default=DEFAULT_URL, help="Filtered UCI listing URL to scrape.")
    parser.add_argument(
        "--take",
        type=int,
        default=25,
        help="Rows requested per page while paginating (default: 25).",
    )
    parser.add_argument(
        "--timeout",
        type=int,
        default=30,
        help="HTTP timeout in seconds (default: 30).",
    )
    args = parser.parse_args([])

    ids = scrape_all_datasets(args.url, take=args.take, timeout=args.timeout)
    if not ids:
        raise SystemExit("No dataset rows were found. The page structure may have changed.")
        return []

    return ids

In [19]:

# extract datasets
ids = get_data()
print(ids)


[53, 45, 186, 17, 222, 2, 320, 352, 109, 19, 697, 350, 144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 602, 27, 20, 336, 242, 601, 292, 174, 327, 545, 80, 42, 267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 529, 579, 936, 46, 942, 848, 563, 890, 52, 878, 383, 225, 850, 572, 887, 857, 445, 158, 151, 101, 571, 145, 938, 484, 915, 880, 864, 365, 110, 547, 759, 244, 193, 732, 763, 33, 105, 76, 143, 451, 176, 426, 603, 198, 12, 16, 264, 379, 760, 471, 39, 212, 461, 172, 591, 467, 90, 536, 50, 503, 47, 419, 161, 81, 117, 357, 485, 911, 312, 58, 342, 537, 827, 30, 329, 582, 799, 247, 40, 43, 26, 565, 380, 149, 146, 13, 38, 270, 372, 148, 913, 367, 229, 257, 277, 713, 300, 54, 28, 83, 728, 722, 22, 567, 184, 78, 107, 95, 63, 755, 3, 70, 44, 75, 69, 91, 23, 74, 32, 96, 82, 18, 88, 8, 147]


In [26]:
# Load Datasets

# Create final dataset obj
algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": []})

for id in ids:
  print(id)
  # load dataset
  dataset = fetch_ucirepo(id=id)
  df = pd.DataFrame(dataset.data.features)
  df['target'] = dataset.data.targets

  # Get dataset features
  name = dataset.metadata['name']
  feature_count = dataset.metadata['num_features']
  instance_count = dataset.metadata['num_instances']
  missing_values = 'Yes' if 'Yes' in dataset.variables['missing_values'] else 'No'

  # Add features to final dataset obj
  algo_selection = pd.concat({'Dataset': name, 'Feature Count': feature_count, 'Instances': instance_count, 'Missing Values': missing_values}, ignore_index=True)

  # Logistic Regression
  logx_train, logx_test, logy_train, logy_test = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=0)
  model = LogisticRegression()
  model.fit(logx_train, logy_train)
  logy_pred = model.predict(logx_test)
  accuracy = accuracy_score(logy_test, logy_pred)

  plt.figure(figsize=(5, 5))
  plt.scatter(logy_test, logy_pred, alpha=0.7)
  plt.xlabel("Actual Values")
  plt.ylabel("Predicted Values")
  plt.title("Actual vs. Predicted Values")
  plt.grid(True)
  plt.show()
  break
  # Neural Network
  nnx_train, nnx_test, nny_train, nny_test = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=0)


  # Decision Tree
  decx_train, decx_test, decy_train, decy_test = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=0)

  # Determine Recommended Algorithm

53


TypeError: cannot concatenate object of type '<class 'str'>'; only Series and DataFrame objs are valid

In [ ]:
# Example Decision Tree
X = [[0, 0], [2, 2]]
y = [0.5, 2.5]
clf = tree.DecisionTreeRegressor()
clf = clf.fit(X, y)
clf.predict([[1, 1]])